# Transform Sellers Data

1. Filter out Invalid rows i.e rows with null seller_id or duplicate seller_id
2. standarize seller_city and seller_state by converting all the city names into lowercase and state names into uppercase
3. write the transformed data to the silver table

In [0]:
#Imports
from pyspark.sql.functions import col,lower,upper,trim

In [0]:
sellers_df=spark.read.table("olist_catalog.bronze.sellers")

### Step1 - Filter out Invalid rows i.e rows with null seller_id or duplicate seller_id

In [0]:
sellers_valid_df = (
    sellers_df.filter(col("seller_id").isNotNull())
        .dropDuplicates(subset=["seller_id"])
)

### Step2 - standarize seller_city and seller_state by converting all the city names into lowercase and state names into uppercase

In [0]:
sellers_final_df = (
    sellers_valid_df.withColumns({'seller_city':lower(trim(col("seller_city"))),
                                  'seller_state':upper(trim(col("seller_state")))})
)

In [0]:
display(sellers_final_df)

### Step3 - write the transformed data to the silver table

In [0]:
(
    sellers_final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.silver.sellers")
)

In [0]:
%sql
select * from olist_catalog.silver.sellers